# How AI Agents Think and Act

## Course Plan:

- Lecture 1 — From LLM to Agent: The Simplest Possible Loop - DONE
- Lecture 2 — Memory and RAG
- Lecture 3 — Graphs and Planning
- Lecture 4 — Multi-Agent Systems

---

Prerequisites:

- API subscription
- .env file with the API key(s)
- tracking token consumption

---
---
---

# Lecture 2 — Memory and State: What the Agent Knows

**Central question:** What happens when the task takes more than one step? What needs to be **remembered**, and where does it live?

Lecture 1 built the agent loop. Every call was self-contained. Now we ask a harder question:
> *What if the agent needs to know something that did not fit in the initial prompt, or something to store between steps?*

We will break the agent deliberately, then fix it with a **vector store** and **retrieval-augmented generation (RAG)**.

---
# Setup

Same stack as Lecture 1, plus **chromadb** — a lightweight vector database that runs entirely in-process.

In [ ]:
# Standard imports and path setup
from typing import Any, List, cast

# Path to access utils.py from the notebook
import os, sys, json, pathlib
sys.path.insert(0, '..')

# OpenAI API types
from openai import OpenAI
from openai.types.chat import (
    ChatCompletion, ChatCompletionMessage,
    ChatCompletionMessageParam, ChatCompletionToolParam,
)

# Vector store - for RAG data
import chromadb
from chromadb.utils import embedding_functions

# Shared helpers from Lecture 1
%load_ext autoreload
%autoreload 2
from shared.utils import get_openai_client, print_messages, to_param, print_tool_call

# Create clients
llm: OpenAI = get_openai_client()
MODEL: str = "gpt-4o-mini"
EMBED_MODEL: str = "text-embedding-3-small"

print("Ready. LLM model:", MODEL, "| Embedding model:", EMBED_MODEL)

<details>
<summary>Details: <strong>What is new in this setup?</strong></summary>

`chromadb` is an open-source vector database. We use its **in-memory** mode (`chromadb.Client()`): no server, no files on disk, the index lives only for the duration of this notebook session. The API is identical to the persistent version — only the constructor differs.

`EMBED_MODEL = "text-embedding-3-small"` is OpenAI's cheapest embedding model: 1,536-dimensional vectors, ~$0.02 per million tokens. Embeddings are a separate API call from completions — they appear on a separate line in your usage bill. Uses the same API key (see .env)

Everything else is carried over from Lecture 1. The agent loop structure does not change — only the tools we give the agent will change.
</details>

---
# Part 1 — The Context Window as Working Memory

- The model is stateless. It has no memory between API calls.
- The only memory it has *during* a conversation is the **message list** we send with every call.
- Technically all message list goes into **one** autoregression completion

Let's show that this works — and measure its cost.

In [ ]:
# Multi-turn conversation: agent "remembers" a fact introduced earlier

messages: List[ChatCompletionMessageParam] = [
    {"role": "system", "content": "You are a helpful math tutor."},
]

# Turn 1 — introduce a fact
messages.append( 
    {"role": "user", "content": "My lucky number is 37. Please remember it."}
)
r1: ChatCompletion = llm.chat.completions.create(model=MODEL, messages=messages, max_tokens=64)
messages.append(to_param(r1.choices[0].message))
print("-------\nTurn 1:", r1.choices[0].message.content)

# Turn 2 — ask about the fact
messages.append(
    {"role": "user", "content": "What is my lucky number cubed?"}
)
r2: ChatCompletion = llm.chat.completions.create(model=MODEL, messages=messages, max_tokens=64)
messages.append(to_param(r2.choices[0].message))
print("-------\nTurn 2:", r2.choices[0].message.content)

# Turn 3 — add another fact, then reference both
messages.append(
    {"role": "user", "content": "My other favorite number is 11. What is the sum of my two numbers?"}
)
r3: ChatCompletion = llm.chat.completions.create(model=MODEL, messages=messages, max_tokens=64)
messages.append(to_param(r3.choices[0].message))
print("------\nTurn 3:", r3.choices[0].message.content)

# Measure prompt token growth across turns
print("------\n")
for i, (r, label) in enumerate([(r1,"Turn 1"),(r2,"Turn 2"),(r3,"Turn 3")], 1):
    print(f"  {label} — prompt tokens: {r.usage.total_tokens} = input {r.usage.prompt_tokens} + output {r.usage.completion_tokens}")

<details>
<summary>Details: <strong>Where is the memory?</strong></summary>

There is no memory *inside* the model. Between API calls, nothing is preserved on OpenAI's side. The agent "remembers" 37 because we re-sent the earlier exchange in `messages`.

Watch the prompt token count grow across turns. Each turn re-sends the **entire conversation history**. Prompt tokens grow linearly with conversation length, while completion tokens stay roughly constant. In a long session, prompt costs dominate.

This in-context memory is called the **context window**. For `gpt-4o-mini` it is 128k tokens — about 90,000 words. That sounds large, but a real knowledge base (documentation, manuals, past conversations) can easily exceed it. We need a different approach for long-term knowledge.

Also note: **context window memory is ephemeral**. When the Python process ends (or the user starts a new session), the message list is gone. Agents that need to remember across sessions must persist the message list externally — in a database, a file, or a vector store.
</details>

---
# Part 2 — The Limits: What the Agent Cannot Know

In-context memory works great for facts introduced *in the current conversation*.
But what about facts that live in our **documents** — outside the model's training data?

Let's ask the model a specific question about facts stored in our `docs/` folder.

In [ ]:
# Ask about specific facts from our documents — without providing them

QUESTION: str = (
    "How many total entries did Ramanujan's notebooks contain, "
    "and on what exact date did G.H. Hardy first receive Ramanujan's letter?"
)

r: ChatCompletion = llm.chat.completions.create(
    model=MODEL,
    messages=cast(List[ChatCompletionMessageParam], [
        {"role": "system", "content": "You are a math history assistant. Answer as precisely as you can."},
        {"role": "user",   "content": QUESTION},
    ]),
    max_tokens=128,
)

print("The answer without document access:")
print(r.choices[0].message.content)

<details>
<summary>Details: <strong>What just happened?</strong></summary>

The model's answer might be approximately correct, partially correct, or confidently wrong — depending on what made it into its training data. The important observation is structural, not factual:

**The model has no access to our documents.** It falls back on learned patterns. It produces the most probable text given the prompt, which may or may not match the specific facts in our files.

Worse: **the model's confidence is not calibrated to its accuracy**. It will write a fluent, assertive answer whether it is correct or not. There is no "I'm not sure about this specific number" signal in the output unless we prompt for it explicitly.

This is the retrieval problem: we have a body of documents we want the agent to reason from. The context window can hold them — but only if we *provide* them. When there are thousands of pages, we cannot send them all. We need to retrieve only what is relevant to the current question.
</details>

---
# Part 3 — A Vector Store as Long-Term Memory

The plan:
1. Load documents from the `docs/` folder
2. Embed each document into a vector using OpenAI's embedding model
3. Store the vectors in Chroma
4. At query time: embed the question, find the nearest document vectors, retrieve the text

First, load the documents.

In [ ]:
# Load all markdown files from the docs/ folder

# Define path to the folder with our documents
docs_dir: pathlib.Path = pathlib.Path("docs")

# Create in-memory collection for the documents
documents: list[dict[str, str]] = []

# Read each file as one document and add in the documents collection
for doc_path in sorted(docs_dir.glob("*.md")):
    text: str = doc_path.read_text(encoding="utf-8")
    documents.append({"id": doc_path.stem, "text": text, "source": doc_path.name})
    print(f"  {doc_path.name}: {len(text)} chars")

print(f"\nTotal: {len(documents)} documents loaded")

Now embed and index all documents in Chroma.

An **embedding** of a piece of text is **a point in a mathematical space** so that similar texts end up close together and different texts end up far apart.

**Spatial intuition:** *We are embedding a piece of text into a geometric space*.

In [ ]:
# Create an embedding function backed by OpenAI
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name=EMBED_MODEL,
)

# In-memory Chroma client — no server, no disk persistence
chroma_client: chromadb.ClientAPI = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="math_docs",
    embedding_function=openai_ef,
)

# Add all documents — Chroma calls the embedding function automatically
collection.add(
    ids=[doc["id"] for doc in documents],
    documents=[doc["text"] for doc in documents],
    metadatas=[{"source": doc["source"]} for doc in documents],
)

print(f"Indexed {collection.count()} documents.")
print(f"Each document is a {EMBED_MODEL} embedding: 1,536-dimensional vector.")

<details>
<summary>Details: <strong>What is an embedding?</strong></summary>

An **embedding** maps a piece of text to a fixed-length vector of floating-point numbers. For `text-embedding-3-small`, that is 1,536 numbers. The key property: **semantically similar texts produce similar vectors** — close together in Euclidean space, high cosine similarity.

"Ramanujan's letter to Hardy" and "Hardy receives the notebook results" land near each other. "Basel problem" lands far away. This is learned geometry: the model was trained so that related concepts are proximal.

**Chroma's job** is to store these vectors and, at query time, find the *k* nearest stored vectors to a query vector. This is approximate nearest-neighbor (ANN) search. The matching is **semantic**, not lexical — a query for "infinite series summation" can surface a document about "summing reciprocals of squares" even without exact keyword overlap.
</details>
<br>
<details>
<summary>Details: <strong>What is stored in Chroma DB?</strong></summary>

**Chroma will store **all three** of the following:**

1. **The embedding vectors:** Stored in a binary format (FAISS, HNSW, or Chroma’s own structure depending on backend). These are the actual points in vector space used for similarity search.

2. **The full document texts:** Stored — unless you explicitly disable text storage. Chroma stores the raw text in its metadata database (SQLite or DuckDB). This is how Chroma can return the original text when you query. You can disable document storage if you pass **documents=None** in **collection.add()** or **metadata={"hnsw:storeRaw": False}** in **chroma_client.get_or_create_collection()**

3. **Metadata (including doc IDs):** Stored as JSON blobs in the same metadata DB. IDs and references can be used for referencing externally stored data - web pages, files on disk, records in you DB

**Chroma DB Can be Persistent or Ephemeral (in-memory)**

Use different Chroma clients (the API is identical): 
- `chromadb.Client()` stores everything in RAM. 
- `chromadb.PersistentClient(path="./chroma_db")` writes to disk. 
</details>

**Computing embeddings has cost: embedding input tokens.**

Let's compute embeddings separately, not relying on Chroma to call our embedding function.

In [ ]:
# Re-create empty chroma collection again
collection = chroma_client.get_or_create_collection(
    name="math_docs",
    embedding_function=None # Note that embedding function is not needed here.
)

# Call the embeddings API directly to capture token usage before indexing
all_texts: list[str] = [doc["text"] for doc in documents]
embed_response = llm.embeddings.create(model=EMBED_MODEL, input=all_texts)
total_embed_tokens: int = embed_response.usage.total_tokens

# Pass pre-computed embeddings to Chroma — avoids a redundant second API call
precomputed_embeddings: list[list[float]] = [item.embedding for item in embed_response.data]
collection.add(
    ids=[doc["id"] for doc in documents],
    documents=all_texts,
    embeddings=precomputed_embeddings,
    metadatas=[{"source": doc["source"]} for doc in documents],
)

print(f"Indexed {collection.count()} documents using embedding model {EMBED_MODEL}.")
print(f"Created {len(embed_response.data)} embedding vectors; each containing {len(embed_response.data[0].embedding)} float numbers.")
print(f"Embedding tokens used: {total_embed_tokens}  (~${total_embed_tokens / 1_000_000 * 0.02:.6f} at $0.02/1M tokens)")

<details>
<summary>Details: <strong>Does Chroma need the embedding function?</strong></summary>


### Chroma is a *vector database*, not an *embedding model*

>> - At its basics it is a geometric memory (vector database).  
>> - You give it vectors, and it gives you back nearby vectors.  
>> - If you want Chroma to accept raw text, you must give it an embedding function.  
>> - If you compute embeddings yourself, Chroma only accepts vectors.

### Embedding function in Chroma is just a *higher-level helper*:

| Chroma configuration | Can you pass text? | Can you pass vectors? |
|--------------------------------|--------------------|-----------------------|
| **With embedding function** | **Yes** | **Yes** - though the usage will be confusing |
| **Without embedding function**  | **No** | **Yes** |

You can keep all embedding computation under your control — which is what you want for:

- token usage tracking  
- consistent embedding model selection  
- reproducibility  
- explicit architecture  

</details>
<br>
<details>
<summary>Details: <strong>Embeddings Deep Dive</strong></summary>

### 1. Intuition: “All possible texts → huge space → many conceptual subspaces”

A general‑purpose embedding model trained on the entire internet ends up with:

- **broad semantic regions** (law, medicine, cooking, math, fiction, etc.)
- **sub‑regions** inside each domain (e.g., within “math”: algebra, topology, probability)
- **fine‑grained neighborhoods** inside those (e.g., within “probability”: Bayesian inference, Markov chains)

Think of it as a **fractal semantic landscape**.
The model learns to place unrelated domains far apart because that reduces training loss.

### 2. Intuition: “Texts from the same domain might become indistinguishable”

This is *partially* true — but the nuance matters.

- **True:** If your domain is extremely specialized (e.g., quantum chemistry, legal contracts, medical imaging), a general embedding model may not capture the **fine‑grained distinctions** you care about.

>> It will still cluster them correctly, but the **resolution** inside that cluster may be too low.

- **Not true:** General embeddings do *not* collapse domain texts into a single blob.  
They still separate them — just not always with the precision a domain expert wants.

>> Think of it like a map:
>> - A general embedding model gives you a **world map**.  
>> - A domain‑specific embedding model gives you a **street map** of your city.

**Both are maps.** One is just more detailed where you care.

### 3. Intuition: “A specialized embedding model trained on domain text would be more efficient”

This is **absolutely correct** — and widely observed in practice.

- **Why domain‑specific embeddings can outperform general ones:**
    - They allocate **more representational capacity** to distinctions *within* the domain.
    - They learn **domain‑specific terminology** and relationships.
    - They compress irrelevant parts of the semantic universe (e.g., pop culture, sports, cooking) and use that capacity for your domain.

This is why biomedical, legal, and financial embeddings exist — they outperform general models on their own turf.

### 4. How embeddings *actually* work (mapped to your intuition)

Let’s map your intuition to the real mechanics:

| Your intuition | How embeddings actually work |
|----------------|------------------------------|
| “All texts live in one huge space” | Yes — the model learns a single high‑dimensional manifold of meaning. |
| “Domains form subspaces” | Yes — clusters and submanifolds naturally emerge during training. |
| “I want to hint the algorithm to distinguish texts better” | You can’t hint at inference time, but you *can* use domain‑specific models or fine‑tuning. |
| “General training might be less efficient” | Correct — general models trade depth for breadth. |
| “Domain models are better for domain tasks” | Correct — they allocate more resolution to the distinctions you care about. |

### 5. The deeper truth

General embedding models *do* contain domain‑specific structure — but they compress it.  
They must represent:

- all human topics  
- all writing styles  
- all intents  
- all languages  
- all genres  

in a fixed number of dimensions (1536 or 3072).

A domain‑specific model can use those same dimensions to represent:

- only legal reasoning  
- only medical terminology  
- only chemistry papers  

So the “resolution” inside that domain is higher.
</details>

### Manual retrieval

Before wiring retrieval into an agent, let's see it work directly: send a query, get back the most relevant document.

In [ ]:
# Query the vector store directly — no agent yet

query: str = "Ramanujan notebooks Hardy letter date"
print(f"Query: '{query}'\n")

# Ask Chroma to execute the array of queries - find all matching documents
results = collection.query(query_texts=[query], n_results=2)
# results is a dict { ids: [str[],...], documents: [str[],...], distances: [float[],...], metadatas: [{}[],...]}
# The outer lists correspond to each query you sent.

for doc_text, meta, distance in zip(
    results["documents"][0], # List of all documents
    results["metadatas"][0], # List of respective metadata
    results["distances"][0], # List of respective distances from the query
):
    print(f"--- source: {meta['source']} (distance: {distance:.4f}) ---")
    print(doc_text[:400]) # First 40 characters of the document
    print()

<details>
<summary>Details: <strong>What is semantic search?</strong></summary>

A traditional inverted-index search (like grep or Elasticsearch BM25) would require the query to share words with the document. "Hardy letter date" would only match documents containing those exact words.

Semantic search **embeds the query into the same vector space as the documents**. The returned result is determined by **cosine similarity** between the query vector and all stored document vectors. Words do not need to match — concepts do.

The `distance` value is **(1 − cosine_similarity (lower = more similar))**. Two identical texts would have distance ≈ 0. Completely unrelated texts approach 1.

**Practical implication:** Users can query in **natural language** without needing to know the exact phrasing used in the documents. This is why RAG works well for question-answering over large corpora — the retrieval step is robust to paraphrase.

**The failure mode:** Semantic similarity is not logical entailment. A document about *Hardy's biography* might score highly for a query about Ramanujan if it mentions him. Always inspect what gets retrieved, especially when the agent gives a wrong answer — the retrieval step is often where the error originates.
</details>
<br>
<details>
<summary>Details: <strong>What this code did step by step</strong></summary>

**1. You send a natural‑language query to Chroma**
```python
query = "Ramanujan notebooks Hardy letter date"
results = collection.query(query_texts=[query], n_results=2)
```

This does **not** call an LLM.  
This is a **pure vector similarity search** inside Chroma.

What happens internally:

1. Chroma embeds your query using the embedding function you configured.
2. It performs a nearest‑neighbor search against stored embeddings.
3. It returns the top `n_results` matches.

The return value is a **dictionary of lists**, structured like this:

```python
{
  "ids": [[...]], 
  "embeddings": [[...]],
  "documents": [[...]],
  "metadatas": [[...]],
  "distances": [[...]]
}
```

Each field is a list of lists because Chroma supports **batch queries**.
You passed one query → so everything is a list containing a single list.

**2. You unpack the results for the first (and only) query**
```python
for doc_text, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
```
</details>

---
# Part 4 — RAG Agent: Retrieval as a Tool

Retrieval works. Now wrap it as a **tool** so the agent can call it on demand.

The agent loop from Lecture 1 is unchanged. What changes is the tool: instead of getting the time or doing math, `search_docs` reaches into the vector store and returns a text excerpt.

In [25]:
# Define a retrieval tool and wrap it for the agent loop

def search_docs(query: str) -> str:
    """Search the document knowledge base for information relevant to the query."""
    # Retrieve top-2 most relevant documents
    results = collection.query(query_texts=[query], n_results=2)

    # Format as plain text for the model to read
    context_parts: list[str] = [
        f"[Source: {meta['source']}]\n{doc_text}"
        for doc_text, meta in zip(results["documents"][0], results["metadatas"][0])
    ]
    print(f"  Retrieved {len(context_parts)} chunks for query: '{query}'")
    return "\n\n---\n\n".join(context_parts)


# Tool schema — the model reads this description to decide when to call the tool
rag_tools: List[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": (
                "Search the knowledge base for mathematical history facts. "
                "Call this before answering any factual question."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A short search query describing the information needed.",
                    }
                },
                "required": ["query"],
            },
        },
    }
]

# Dispatch table: tool name → Python function
RAG_FUNCTIONS: dict[str, Any] = {
    "search_docs": lambda args: search_docs(args["query"]),
}

print("RAG tool registered.")

RAG tool registered.


In [26]:
# Agent loop — same pattern as Lecture 1

def run_rag_agent(
    user_message: str,
    tools: List[ChatCompletionToolParam],
    functions: dict[str, Any],
    use_retrieval_prompt: bool = True,
) -> str:
    """Run the agent loop. Returns the model's final text response."""

    # System prompt includes retrieval instruction only when tools are available
    system_content: str = (
        "You are a math history assistant. Always call search_docs before answering factual questions."
        if use_retrieval_prompt
        else "You are a math history assistant. Answer as precisely as you can."
    )

    # Initilize the conversation
    agent_messages: List[ChatCompletionMessageParam] = [
        {"role": "system", "content": system_content},
        {"role": "user",   "content": user_message},
    ]

    # Run a loop of executing the rag tool until LLM returns "stop"
    for turn in range(8):
        print(f"--- Turn {turn + 1} ---")

        # Build API call — omit tools parameter when list is empty
        api_params: dict[str, Any] = {"model": MODEL, "messages": agent_messages}
        if tools:
            api_params["tools"] = tools

        # THINK - make a request to LLM
        response: ChatCompletion = llm.chat.completions.create(**api_params)
        msg: ChatCompletionMessage = response.choices[0].message
        finish: str | None = response.choices[0].finish_reason
        print(f"Finish reason: {finish}")
        agent_messages.append(to_param(msg))

        # Exit when the model when LLM does not need calling tools anymore
        if finish == "stop":
            return msg.content or ""

        # ACT - call the RAG tool per LLM request
        if finish == "tool_calls" and msg.tool_calls:
            for tc in msg.tool_calls:
                print_tool_call(tc)
                args: dict[str, Any] = json.loads(tc.function.arguments)
                result: str = functions[tc.function.name](args)

                # OBSERVE
                agent_messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            continue

        break

    return "[max_turns reached]"

In [27]:
# Run with retrieval — the agent will call search_docs
print("=== WITH RETRIEVAL ===")
# QUESTION was defined above for manual RAG request; use it now with the agent
answer_with_rag: str = run_rag_agent(QUESTION, rag_tools, RAG_FUNCTIONS, use_retrieval_prompt=True)
print()
print("Answer:")
print(answer_with_rag)

=== WITH RETRIEVAL ===
--- Turn 1 ---
Finish reason: tool_calls
  Function: search_docs Args: {
    "query": "total entries in Ramanujan's notebooks"
}
  Retrieved 2 chunks for query: 'total entries in Ramanujan's notebooks'
  Function: search_docs Args: {
    "query": "G.H. Hardy received Ramanujan's letter date"
}
  Retrieved 2 chunks for query: 'G.H. Hardy received Ramanujan's letter date'
--- Turn 2 ---
Finish reason: stop

Answer:
Srinivasa Ramanujan's notebooks contain approximately **3,909 entries** comprising theorems, formulas, and identities, most presented without proof. 

G.H. Hardy first received Ramanujan's letter on **January 16, 1913**. This nine-page letter contained 120 statements of theorems, which Hardy later described as one of the most remarkable letters he had ever received.


<details>
<summary>Details: <strong>What did the agent actually do?</strong></summary>

The agent loop is structurally identical to Lecture 1. What changed is the *tool*: instead of calling `get_current_time()` or `calculate()`, it calls `search_docs()`, which queries the Chroma vector store and returns relevant document text.

The flow:
1. **Turn 1**: model reads the system prompt + question, sees the `search_docs` tool schema, decides to call it
2. **Tool execution**: `search_docs(query)` → Chroma returns top-2 matching documents as text
3. **Turn 2**: model reads the original question **plus** the retrieved document text, now has the specific facts, generates the correct answer

This is RAG. The model's generation is **augmented** by **retrieved** context. The model does not memorize facts — it uses the tool to look them up on demand.

**The system prompt does real work here.** "Always call search_docs before answering factual questions" is not code logic — it is a natural-language instruction. The model may comply or not, depending on how it interprets the prompt. If you remove it, the model will sometimes answer from memory and sometimes retrieve, producing inconsistent behavior. The **reliability of the agent depends on the reliability of the prompt**.
</details>
<br>
<details>
<summary>Details: <strong>Chunking: why this demo skips it and why production systems don't</strong></summary>

Our documents are short (~300–500 words each) so we store each file as a single Chroma entry. In production, documents are long — research papers, manuals, transcripts — and a single embedding per document discards structure.

The standard practice is **chunking**: split each document into overlapping windows of ~300–500 tokens, embed each chunk separately, and store chunks instead of full documents. At retrieval time, you retrieve the most relevant *chunks*, not whole documents.

**Why overlapping?** A fact might straddle a chunk boundary. Overlapping windows (e.g., 50-token overlap between adjacent 300-token chunks) ensure no sentence is split across two non-retrieved chunks.

**What to pass to the model?** Retrieved chunks, plus optionally a few tokens of surrounding context to avoid jarring mid-sentence starts.

LangChain's `RecursiveCharacterTextSplitter` and LlamaIndex's `SentenceSplitter` implement common chunking strategies. For this lecture, we skip chunking to keep the code focused on the retrieval concept.
</details>

---
# Part 5 — The Contrast: Disable Retrieval

Now run the **same agent, same question** — but without the `search_docs` tool.

The model falls back on its training data. We compare the two answers side by side.

In [28]:
# Run without retrieval tool — model answers from training data alone
print("=== WITHOUT RETRIEVAL ===")
answer_without_rag: str = run_rag_agent(
    QUESTION,
    tools=[],       # no tools available
    functions={},
    use_retrieval_prompt=False,
)

# Side-by-side comparison
print()
print("=" * 60)
print("QUESTION:", QUESTION)
print()
print("WITH RAG:")
print(answer_with_rag)
print()
print("WITHOUT RAG:")
print(answer_without_rag)

=== WITHOUT RETRIEVAL ===
--- Turn 1 ---
Finish reason: stop

QUESTION: How many total entries did Ramanujan's notebooks contain, and on what exact date did G.H. Hardy first receive Ramanujan's letter?

WITH RAG:
Srinivasa Ramanujan's notebooks contain approximately **3,909 entries** comprising theorems, formulas, and identities, most presented without proof. 

G.H. Hardy first received Ramanujan's letter on **January 16, 1913**. This nine-page letter contained 120 statements of theorems, which Hardy later described as one of the most remarkable letters he had ever received.

WITHOUT RAG:
Ramanujan's notebooks contained a total of 87 entries. G.H. Hardy first received Ramanujan's letter on January 16, 1913.


<details>
<summary>Details: <strong>Why does this matter beyond getting the right number?</strong></summary>

The model's answer without retrieval might be:
- **Approximately correct** — if this fact appears prominently in its training corpus
- **Confidently wrong** — a plausible-sounding number that differs from the document
- **Hedged** — "I believe it was around 3,000" — which is honest but unhelpful for a precise question

All three are failures for an agent that is supposed to reason from *your* documents. The structural problem is the same in each case: the model has no ground truth to check against.

**RAG does not eliminate hallucination.** The model can still misread retrieved context, combine facts incorrectly, or add details not present in the source. But it shifts the failure mode: instead of inventing facts from nothing, it is now constrained by what the retrieved text actually says. That is an auditable failure — you can check the document.

**The real lesson**: the quality of an agent's answers is bounded by the quality of its retrieval. A good language model with bad retrieval gives bad answers. The embedding model, the chunking strategy, and the number of retrieved chunks matter as much as the LLM itself.
</details>

<details>
<summary>Details: <strong>Types of agent memory</strong></summary>

What we have built illustrates two distinct memory types. Researchers and practitioners distinguish several:

| Type | Mechanism | Lifespan | Example |
|------|-----------|----------|---------|
| **In-context (working)** | Message list | Current session | The user's lucky number from Part 1 |
| **External (episodic/semantic)** | Vector store + retrieval | Persistent | Our `docs/` knowledge base |
| **Procedural** | System prompt + tool definitions | Fixed per deployment | "Always call search_docs first" |
| **Parametric** | Model weights | Until fine-tuning | Everything the model learned from training |

RAG primarily augments **semantic memory** — factual knowledge about the world — without touching model weights. Fine-tuning updates parametric memory, but is expensive and can cause forgetting. RAG is preferred when facts change over time or when the source of truth must be auditable.

Lecture 3 will introduce **state** as a first-class concept in LangGraph — a more structured way to manage what the agent knows at each step of a complex pipeline.
</details>

---
# Part 6 — Chunking: Retrieval at Finer Granularity

So far each document is one Chroma entry. That works for our short files.

In reality documents are long — and a single embedding per document loses internal structure. The fix is **chunking**: split each document into small, overlapping windows and embed each chunk separately.

We will chunk one document by hand, index the chunks, and compare what gets retrieved.

In [ ]:
# Split documents into overlapping chunks and index them

def make_chunks(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into overlapping windows of characters."""
    chunks: list[str] = []
    start: int = 0
    while start < len(text):
        end: int = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        # Advance by (chunk_size - overlap) so adjacent chunks share overlap chars
        start += chunk_size - overlap
    return chunks


CHUNK_SIZE: int = 400   # characters per chunk
OVERLAP: int = 80       # characters shared between adjacent chunks

# Build a flat list of (chunk_id, chunk_text, source) across all documents
chunk_ids: list[str] = []
chunk_texts: list[str] = []
chunk_metas: list[dict] = []

for doc in documents:
    chunks: list[str] = make_chunks(doc["text"], CHUNK_SIZE, OVERLAP)
    for i, chunk in enumerate(chunks):
        chunk_ids.append(f"{doc['id']}__chunk{i}")
        chunk_texts.append(chunk)
        chunk_metas.append({"source": doc["source"], "chunk": i})
    print(f"  {doc['source']}: {len(doc['text'])} chars → {len(chunks)} chunks")

print(f"
Total chunks: {len(chunk_ids)}  (vs {len(documents)} whole-doc entries before)")

# Embed all chunks and count tokens
chunk_embed_response = llm.embeddings.create(model=EMBED_MODEL, input=chunk_texts)
chunk_embed_tokens: int = chunk_embed_response.usage.total_tokens
chunk_embeddings: list[list[float]] = [item.embedding for item in chunk_embed_response.data]
print(f"Embedding tokens used for chunks: {chunk_embed_tokens}")

# Index chunks in a separate Chroma collection
chunk_collection = chroma_client.get_or_create_collection(
    name="math_docs_chunked",
    embedding_function=openai_ef,
)
chunk_collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    embeddings=chunk_embeddings,
    metadatas=chunk_metas,
)
print(f"Indexed {chunk_collection.count()} chunks.")

In [ ]:
# Compare whole-doc vs. chunk retrieval for the same query

CHUNK_QUERY: str = "exact date Hardy received Ramanujan letter"

# Retrieve from the whole-document collection
whole_results = collection.query(query_texts=[CHUNK_QUERY], n_results=1)
whole_hit = whole_results["documents"][0][0]
whole_dist = whole_results["distances"][0][0]

# Retrieve from the chunked collection
chunk_results = chunk_collection.query(query_texts=[CHUNK_QUERY], n_results=1)
chunk_hit = chunk_results["documents"][0][0]
chunk_dist = chunk_results["distances"][0][0]
chunk_source = chunk_results["metadatas"][0][0]

print(f"Query: '{CHUNK_QUERY}'
")
print(f"=== WHOLE-DOC retrieval (distance: {whole_dist:.4f}) ===")
print(whole_hit[:400])
print()
print(f"=== CHUNK retrieval (distance: {chunk_dist:.4f}, source: {chunk_source['source']} chunk {chunk_source['chunk']}) ===")
print(chunk_hit)

<details>
<summary>Details: <strong>Why chunking changes retrieval quality</strong></summary>

When a document is indexed as a single unit, its embedding is an average over all the content — the broad topic of the document. A query about one specific sentence competes against the entire document's meaning.

With chunking, each chunk's embedding reflects only its local content. A query about Hardy's letter date lands closer to the chunk that *contains* that sentence, because nothing else dilutes the vector.

**The overlap is not decoration.** A key sentence at the boundary between two chunks might be split mid-thought. The overlap window ensures every sentence appears fully in at least one chunk.

**Chunk size is a hyperparameter.** Smaller chunks → higher retrieval precision, but the model gets less surrounding context. Larger chunks → more context passed to the model, but noisier retrieval. Typical production values: 256–512 tokens per chunk, 10–20% overlap.

**Character splitting is a simplification.** Production systems split on sentence or paragraph boundaries to avoid cutting a sentence mid-word. LangChain's  tries paragraph → sentence → word boundaries in order, falling back to character splitting only when needed.
</details>

---
# Summary

| Part | What we built | Key insight |
|------|--------------|-------------|
| 1 | Multi-turn in-context memory | Context window = working memory. Stateless model, stateful client. Tokens grow each turn. |
| 2 | Knowledge gap | Model generates plausible text even without reliable facts. Confidence ≠ accuracy. |
| 3 | Vector store | Text → embedding → nearest-neighbor search. Semantic, not lexical. |
| 4 | RAG agent | Retrieval as a tool: agent decides when to look things up, then reasons from the retrieved text. |
| 5 | Contrast | Disable retrieval → hallucination. The tool is the grounding. |
| 6 | Chunking | Finer granularity → higher retrieval precision. Chunk size and overlap are hyperparameters. |

> *The agent's "knowledge" is whatever is in the context window at call time.*  
> *Long-term memory requires an external store and an explicit retrieval step.*

The interface between the agent and the vector store is — again — a string: the query. The interface between retrieval and the model is — again — a string: the retrieved text injected into the message list. **Strings flow between every component.**

---
### Discussion Prompts

1. What determines whether the model calls `search_docs`? What could make it skip retrieval even when it should retrieve?
2. We retrieve 2 documents. What happens if the correct answer requires combining facts from 4 documents? What if the most relevant document ranks 3rd?
3. In-context memory is lost when the session ends. How would you persist it? What would you store — the full message list, a summary, or something else?
4. The model can hallucinate even *after* retrieval (misread the retrieved text). How would you detect this?